# Inteligencia de Negocios

## Componente Práctico Semana 1

## Grupo # 3

### ALEJANDRA BEATRIZ TELLO GONZÁLEZ
### PABLO RAMIRO VALLEJO ZÚÑIGA


---
## Fase I – Configuración

### Dominio de negocio: **Ciberseguridad Bancaria**

El proyecto trabaja con datos del área de seguridad de la información en el sector financiero.  
Las tres fuentes de datos seleccionadas cubren distintas dimensiones de la operación de seguridad:

| # | Fuente | Tipo | Descripción |
|---|--------|------|-------------|
| 1 | `api_security_events` | PostgreSQL | Eventos de seguridad en canales bancarios digitales |
| 2 | `incidentes_seguridad.csv` | CSV | Registro de incidentes de red y ataques detectados |
| 3 | `vulnerabilidades_cve.json` | JSON | Resultado de escaneo de vulnerabilidades (CVE) |

### Infraestructura
- **Docker + PostgreSQL 17**: contenedor `ciber_12B` levantado con `docker-compose.yml`
- **Variables de entorno**: gestionadas con `python-dotenv` desde el archivo `.env`
- **IDE**: PyCharm Pro con environment virtual `Practica_G3`

---
## Fase II – Desarrollo

### Importación de dependencias

Se importan las librerías necesarias para:
- **pandas**: manipulación y análisis de DataFrames
- **sqlalchemy + psycopg2**: conexión ORM a PostgreSQL
- **json**: lectura del archivo de vulnerabilidades
- **dotenv**: carga de credenciales desde `.env` (buena práctica de seguridad)

In [1]:
import pandas as pd
import json
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()
print("Dependencias cargadas correctamente")

Dependencias cargadas correctamente


In [ ]:
%%sql


### Manejo de variables de entorno

Las credenciales de la base de datos se leen del archivo `.env` para evitar exponer datos sensibles en el código fuente.  
Esta práctica es obligatoria en entornos bancarios donde los repositorios pueden ser auditados.

In [2]:
DB_USER     = os.getenv("POSTGRES_USER")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD")
DB_NAME     = os.getenv("POSTGRES_DB")
DB_HOST     = os.getenv("POSTGRES_HOST")
DB_PORT     = os.getenv("POSTGRES_PORT", "5432")

print(f"🔌 Conectando a: {DB_HOST}:{DB_PORT}/{DB_NAME} como usuario '{DB_USER}'")

🔌 Conectando a: localhost:5432/dbmcib12b como usuario 'admin'


### Conexión y carga de la base de datos PostgreSQL

Se crea el engine de SQLAlchemy apuntando al contenedor Docker.
Luego se ejecuta el script SQL de seed para poblar la tabla `api_security_events`,
que contiene eventos de seguridad registrados en los canales bancarios digitales.

In [3]:
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# Cargar el script SQL de seed (crear tabla + insertar 150 registros)
with open("data/seed_api_security.sql", "r") as f:
    sql_seed = f.read()

with engine.connect() as conn:
    for statement in sql_seed.split(";"):
        stmt = statement.strip()
        if stmt and not stmt.startswith("--"):
            conn.execute(text(stmt))
    conn.commit()

print("Tabla 'api_security_events' creada y datos cargados")

Tabla 'api_security_events' creada y datos cargados


---
## Generación de los tres DataFrames principales

En esta sección se extraen los datos de cada una de las tres fuentes definidas en la Fase I.
Cada DataFrame representa una vista diferente del ecosistema de ciberseguridad:

- `df_postgres` → eventos de seguridad en APIs bancarias (fuente: PostgreSQL)
- `df_csv` → incidentes de red detectados por el SOC (fuente: CSV)
- `df_json` → vulnerabilidades CVE identificadas en escaneos (fuente: JSON)

In [4]:
# ── DataFrame 1: desde PostgreSQL ──────────────────────────────────────────
df_postgres = pd.read_sql("SELECT * FROM api_security_events", engine)

# ── DataFrame 2: desde CSV ─────────────────────────────────────────────────
df_csv = pd.read_csv("data/incidentes_seguridad.csv")

# ── DataFrame 3: desde JSON ────────────────────────────────────────────────
with open("data/vulnerabilidades_cve.json", "r", encoding="utf-8") as f:
    vulnerabilidades = json.load(f)
df_json = pd.DataFrame(vulnerabilidades)

print(f"df_postgres : {df_postgres.shape[0]} filas × {df_postgres.shape[1]} columnas")
print(f"df_csv      : {df_csv.shape[0]} filas × {df_csv.shape[1]} columnas")
print(f"df_json     : {df_json.shape[0]} filas × {df_json.shape[1]} columnas")

df_postgres : 150 filas × 12 columnas
df_csv      : 300 filas × 14 columnas
df_json     : 80 filas × 13 columnas


---
## Script de Análisis 1 – API Security Events (PostgreSQL)

Este DataFrame contiene **150 eventos** de seguridad registrados en los canales digitales del banco  
(Cash Management, Mobile Banking, Internet Banking, etc.).  
Es la fuente más crítica porque refleja el comportamiento real de las APIs expuestas al cliente.

In [5]:
# ── i. Primeras 5 filas ────────────────────────────────────────────────────
# Se visualiza la estructura inicial del DataFrame para validar que la extracción
# desde PostgreSQL fue exitosa y los tipos de datos son correctos.
print("── Primeras 5 filas ──")
df_postgres.head()

── Primeras 5 filas ──


,event_id,event_date,channel,event_type,http_method,endpoint,response_code,response_time_ms,risk_score,resolved,user_agent,country_code
0,1,2024-01-01,Internet Banking,Auth Failure,GET,/api/v1/transactions,429,1223,4,True,Mozilla/5.0,EC
1,2,2024-01-03,Cash Management,Token Expired,DELETE,/api/v1/transactions,200,344,3,False,Mozilla/5.0,CN
2,3,2024-01-05,Internet Banking,Rate Limit Exceeded,GET,/api/v2/auth,500,4944,3,True,nan,EC
3,4,2024-01-07,Cash Management,Anomalous Pattern,DELETE,/api/v1/balance,429,2459,2,True,nan,EC
4,5,2024-01-09,Internet Banking,Token Expired,DELETE,/api/v1/transactions,400,3321,2,False,Mozilla/5.0,EC


In [6]:
# ── ii. Columnas con valores nulos ─────────────────────────────────────────
# En un entorno bancario, los nulos pueden indicar datos de transacciones incompletas
# o eventos donde el agente HTTP no fue capturado (ej: tráfico automatizado).
nulos_pg = df_postgres.isnull().any()
print("Columnas con valores nulos:")
print(nulos_pg[nulos_pg == True])

Columnas con valores nulos:
Series([], dtype: bool)


In [7]:
# ── iii. Conteo de valores faltantes por columna ───────────────────────────
# Permite priorizar la limpieza de datos: columnas con muchos nulos
# requieren estrategias de imputación antes de la fase de transformación ETL.
print("Valores faltantes por columna:")
print(df_postgres.isnull().sum()[df_postgres.isnull().sum() > 0])

Valores faltantes por columna:
Series([], dtype: int64)


In [8]:
# ── iv. Estadísticas de variable numérica: response_time_ms ───────────────
# El tiempo de respuesta es un KPI de seguridad: tiempos elevados pueden indicar
# ataques de DoS, consultas inyectadas o sobrecarga del API Gateway.
stats_rt = df_postgres["response_time_ms"].agg(["mean", "max", "min"])
print("Estadísticas de tiempo de respuesta (ms):")
print(f"  Promedio : {stats_rt['mean']:.2f} ms")
print(f"  Máximo   : {stats_rt['max']} ms")
print(f"  Mínimo   : {stats_rt['min']} ms")

Estadísticas de tiempo de respuesta (ms):
  Promedio : 2466.21 ms
  Máximo   : 4944.0 ms
  Mínimo   : 71.0 ms


In [9]:
# ── v. Valor máximo y mínimo de risk_score agrupado por channel y event_type
# Identifica qué combinación de canal + tipo de evento presenta mayor y menor riesgo.
# Información clave para priorizar controles de seguridad por canal.
riesgo_por_grupo = (
    df_postgres
    .groupby(["channel", "event_type"])["risk_score"]
    .agg(["max", "min"])
    .reset_index()
    .rename(columns={"max": "risk_max", "min": "risk_min"})
    .sort_values("risk_max", ascending=False)
)
print("Máximo y mínimo de risk_score por canal y tipo de evento:")
print(riesgo_por_grupo.to_string(index=False))

Máximo y mínimo de risk_score por canal y tipo de evento:
         channel            event_type  risk_max  risk_min
     ATM Network          Auth Failure         5         1
 Cash Management   Rate Limit Exceeded         5         1
 Cash Management Unauthorized Endpoint         5         5
     ATM Network Unauthorized Endpoint         5         2
Open Banking API         Token Expired         5         2
Open Banking API Unauthorized Endpoint         5         3
    POS Terminal     Anomalous Pattern         5         1
 Cash Management     Injection Attempt         5         2
  Mobile Banking     Anomalous Pattern         5         3
Internet Banking   Rate Limit Exceeded         5         2
  Mobile Banking Unauthorized Endpoint         5         1
  Mobile Banking     Injection Attempt         5         3
Open Banking API     Anomalous Pattern         5         1
    POS Terminal          Auth Failure         5         1
     ATM Network     Anomalous Pattern         4         

---
## Script de Análisis 2 – Incidentes de Seguridad de Red (CSV)

Este DataFrame contiene **300 incidentes** registrados por el sistema de detección de intrusos (IDS).  
Incluye metadatos de red como protocolo, bytes transferidos, tipo de ataque y severidad.  
Es útil para analizar patrones de ataque y evaluar la efectividad de los controles perimetrales.

In [10]:
# ── i. Primeras 5 filas ────────────────────────────────────────────────────
# Se verifica que el CSV fue leído correctamente con todos sus campos.
# Importante confirmar que 'timestamp' se cargó como string (se tipará en ETL).
print("── Primeras 5 filas ──")
df_csv.head()

── Primeras 5 filas ──


,incident_id,timestamp,attack_type,protocol,source_ip,destination_port,packets_sent,bytes_transferred,duration_seconds,severity,status,affected_system,analyst_assigned,false_positive
0,1,2024-01-01 00:00:00,Ransomware,HTTP,192.168.154.202,3306,25017,8071948,2,High,Blocked,Web Server,Ana Torres,False
1,2,2024-01-01 12:00:00,Brute Force,UDP,192.168.90.254,22,31295,6844556,2273,Critical,Blocked,Web Server,Luis Mora,True
2,3,2024-01-02 00:00:00,Phishing,HTTP,192.168.162.115,3306,21666,4946193,2849,Medium,Blocked,DB Server,Alejandra T.,False
3,4,2024-01-02 12:00:00,Ransomware,HTTP,192.168.105.135,21,10016,6734239,1769,Critical,Blocked,Auth Server,Luis Mora,True
4,5,2024-01-03 00:00:00,XSS,HTTPS,192.168.196.58,8080,20906,8196834,1950,High,Detected,API Gateway,Luis Mora,False


In [11]:
# ── ii. Columnas con valores nulos ─────────────────────────────────────────
# La columna 'analyst_assigned' puede tener nulos cuando el incidente
# fue manejado automáticamente por el SIEM sin intervención humana.
nulos_csv = df_csv.isnull().any()
print("Columnas con valores nulos:")
print(nulos_csv[nulos_csv == True])

Columnas con valores nulos:
analyst_assigned    True
dtype: bool


In [12]:
# ── iii. Conteo de valores faltantes por columna ───────────────────────────
# Un analista no asignado (~10% esperado) no es un error: indica automatización.
# En la fase Transform se decidirá si imputar con 'Sistema Automático' o dejar nulo.
print("Valores faltantes por columna:")
total_nulos = df_csv.isnull().sum()
print(total_nulos[total_nulos > 0])

Valores faltantes por columna:
analyst_assigned    30
dtype: int64


In [13]:
# ── iv. Estadísticas de variable numérica: bytes_transferred ──────────────
# El volumen de datos transferidos es un indicador de exfiltración de información.
# Valores máximos muy elevados pueden corresponder a ataques de exfiltración masiva.
stats_bytes = df_csv["bytes_transferred"].agg(["mean", "max", "min"])
print("Estadísticas de bytes transferidos:")
print(f"  Promedio : {stats_bytes['mean']:,.0f} bytes ({stats_bytes['mean']/1024:.1f} KB)")
print(f"  Máximo   : {stats_bytes['max']:,} bytes ({stats_bytes['max']/1048576:.2f} MB)")
print(f"  Mínimo   : {stats_bytes['min']:,} bytes")

Estadísticas de bytes transferidos:
  Promedio : 5,288,476 bytes (5164.5 KB)
  Máximo   : 10,483,005.0 bytes (10.00 MB)
  Mínimo   : 165,077.0 bytes


In [14]:
# ── v. Máximo y mínimo de duration_seconds agrupado por attack_type y severity
# Ataques de larga duración con severidad Critical son los más peligrosos.
# Esta vista ayuda al SOC a calibrar los umbrales de alerta por tipo de ataque.
duracion_grupo = (
    df_csv
    .groupby(["attack_type", "severity"])["duration_seconds"]
    .agg(["max", "min"])
    .reset_index()
    .rename(columns={"max": "duracion_max_seg", "min": "duracion_min_seg"})
    .sort_values("duracion_max_seg", ascending=False)
)
print("Duración máxima y mínima (segundos) por tipo de ataque y severidad:")
print(duracion_grupo.to_string(index=False))

Duración máxima y mínima (segundos) por tipo de ataque y severidad:
  attack_type severity  duracion_max_seg  duracion_min_seg
     Phishing      Low              3596              2199
   Ransomware Critical              3592               459
  Brute Force   Medium              3569                14
         MITM      Low              3500               620
          XSS   Medium              3486                 2
     Phishing     High              3467               106
     Phishing   Medium              3462               404
SQL Injection     High              3454                35
          XSS      Low              3433               143
   Ransomware   Medium              3424                22
   Ransomware      Low              3390               159
         DDoS   Medium              3378               282
  Brute Force Critical              3375               428
  Brute Force      Low              3348               957
         DDoS      Low              3290       

---
## Script de Análisis 3 – Vulnerabilidades CVE (JSON)

Este DataFrame contiene **80 vulnerabilidades** identificadas en el último escaneo de infraestructura.  
Cada registro incluye el CVE ID, score CVSS, sistema afectado y estado del parche.  
En el sector bancario, el patch management tiene plazos regulatorios definidos por la SB (Superintendencia de Bancos).

In [15]:
# ── i. Primeras 5 filas ────────────────────────────────────────────────────
# Se verifica la estructura del JSON normalizado.
# El campo 'patch_applied' puede contener None (nulo) para sistemas sin diagnóstico.
print("── Primeras 5 filas ──")
df_json.head()

── Primeras 5 filas ──


,vuln_id,cve_id,system,vendor,vulnerability_type,cvss_score,cvss_severity,patch_available,patch_applied,discovery_date,remediation_deadline_days,affected_hosts,exploited_in_wild
0,1,CVE-2024-29772,Apache 2.4,Apache Foundation,Denial of Service,5.2,Medium,True,True,2024-09-07,7,6,False
1,2,CVE-2024-19156,Python 3.8,PostgreSQL Global Dev Group,Buffer Overflow,3.9,Low,True,True,2024-10-19,90,4,False
2,3,CVE-2023-82963,nginx 1.18,PostgreSQL Global Dev Group,Remote Code Execution,8.9,High,True,True,2024-02-19,15,24,True
3,4,CVE-2022-18229,Python 3.8,OpenSSL Project,Information Disclosure,6.5,Medium,False,False,2024-10-15,30,20,False
4,5,CVE-2023-41994,nginx 1.18,Python Software Foundation,Information Disclosure,2.7,Low,True,False,2024-10-03,7,33,False


In [16]:
# ── ii. Columnas con valores nulos ─────────────────────────────────────────
# Los nulos en 'patch_applied' indican vulnerabilidades donde el estado
# del parche no ha sido verificado aún — requieren seguimiento urgente.
nulos_json = df_json.isnull().any()
print("Columnas con valores nulos:")
print(nulos_json[nulos_json == True])

Columnas con valores nulos:
patch_applied    True
dtype: bool


In [17]:
# ── iii. Conteo de valores faltantes por columna ───────────────────────────
# Se cuantifica cuántas vulnerabilidades tienen estado de parche desconocido.
# En auditorías PCI-DSS, esto cuenta como hallazgo de control.
print("Valores faltantes por columna:")
total_nulos_json = df_json.isnull().sum()
print(total_nulos_json[total_nulos_json > 0])

Valores faltantes por columna:
patch_applied    25
dtype: int64


In [18]:
# ── iv. Estadísticas de variable numérica: cvss_score ─────────────────────
# El CVSS (Common Vulnerability Scoring System) va de 0 a 10.
# Un promedio > 7 indica una infraestructura con exposición alta.
# El máximo y mínimo permiten identificar los extremos del portafolio de riesgo.
stats_cvss = df_json["cvss_score"].agg(["mean", "max", "min"])
print("Estadísticas del score CVSS:")
print(f"  Promedio : {stats_cvss['mean']:.2f} / 10")
print(f"  Máximo   : {stats_cvss['max']} / 10")
print(f"  Mínimo   : {stats_cvss['min']} / 10")

Estadísticas del score CVSS:
  Promedio : 5.86 / 10
  Máximo   : 10.0 / 10
  Mínimo   : 2.1 / 10


In [19]:
# ── v. Máximo y mínimo de affected_hosts agrupado por system y cvss_severity
# Permite priorizar el parcheo: vulnerabilidades con severidad Critical
# que afectan más hosts deben resolverse primero (ISO 27001 / Ley de Defensa del Consumidor).
hosts_grupo = (
    df_json
    .groupby(["system", "cvss_severity"])["affected_hosts"]
    .agg(["max", "min"])
    .reset_index()
    .rename(columns={"max": "hosts_max", "min": "hosts_min"})
    .sort_values("hosts_max", ascending=False)
)
print("Hosts afectados (máx/mín) por sistema y severidad CVSS:")
print(hosts_grupo.to_string(index=False))

Hosts afectados (máx/mín) por sistema y severidad CVSS:
       system cvss_severity  hosts_max  hosts_min
   Apache 2.4        Medium         50          5
   Django 3.2           Low         48         19
   Python 3.8           Low         48          4
   Node.js 14        Medium         47         10
   Django 3.2        Medium         45          2
   Django 3.2          High         44          5
PostgreSQL 13           Low         44         26
   Apache 2.4           Low         43          7
   Python 3.8        Medium         42          9
  OpenSSL 1.1           Low         40         11
   Node.js 14           Low         39         29
PostgreSQL 13        Medium         39         12
   Node.js 14      Critical         35          2
   nginx 1.18           Low         33         18
   Django 3.2      Critical         32         17
   Python 3.8          High         30         30
  OpenSSL 1.1      Critical         30          6
   nginx 1.18      Critical         26      

---
# Aplicación Profesional de la Práctica
## Alejandra Beatriz Tello González

Me desempeño como Líder de Seguridad de la Información en el sector bancario ecuatoriano, con enfoque específico en pruebas de penetración a APIs y cumplimiento normativo. Esta práctica me resultó particularmente relevante porque conecta directamente con los desafíos técnicos que enfrento en mi trabajo diario.

En el entorno bancario se generan y gestionan datos de naturaleza muy diversa: registros de transacciones financieras, logs de autenticación en canales digitales (banca móvil, Cash Management, Open Banking), eventos de seguridad capturados por el SIEM, resultados de escaneos de vulnerabilidades, alertas del WAF y del IDS/IPS, y métricas de disponibilidad de servicios críticos. Todos estos flujos producen volúmenes masivos de información que históricamente han sido analizados de forma aislada, sin una visión integrada que permita correlacionar eventos entre capas.

La combinación de PostgreSQL, Docker y Python que trabajamos en esta práctica ofrece una respuesta concreta a esa fragmentación. PostgreSQL permite centralizar los eventos de seguridad en una estructura relacional auditada y persistente, ideal para soportar consultas regulatorias (SB, PCI-DSS). Docker garantiza que el entorno de análisis sea reproducible en cualquier nodo del equipo de seguridad sin conflictos de dependencias, lo que es crítico cuando se trabaja con analistas de distintos perfiles. Python, con pandas y SQLAlchemy, proporciona la flexibilidad para transformar datos de fuentes heterogéneas —logs en JSON, reportes en CSV, consultas directas a base de datos— en DataFrames unificados y procesables.

Implementar un proceso ETL en Banco General Rumiñahui, donde actualmente realizo trabajo de seguridad, aportaría beneficios concretos: consolidar en un solo repositorio los eventos del WAF, los resultados del pentesting a APIs y los logs de Cash Management eliminaría el análisis manual en silos que hoy consume horas de trabajo. La fase de transformación permitiría estandarizar formatos de timestamp, clasificar eventos por severidad CVSS y cruzar incidentes de red con el inventario de vulnerabilidades activas.

Las decisiones que podrían tomarse con esta información son de alto impacto: priorizar el parcheo de sistemas con vulnerabilidades Critical explotadas actualmente, detectar patrones de ataques de fuerza bruta correlacionados con horarios de menor monitoreo, identificar endpoints API con tiempos de respuesta anómalos que podrían indicar inyección en curso, y justificar con datos cuantificables la inversión en controles adicionales ante la Gerencia de Riesgos. En un sector donde una brecha de seguridad implica sanciones regulatorias, pérdida de confianza del cliente y potencial daño reputacional, la capacidad de analizar datos de seguridad de forma automatizada e integrada no es una ventaja competitiva: es una necesidad operativa.

## Pablo Vallejo Zúñiga

Actualmente me desempeño como Director de Tecnología de la Prefectura de Loja, y la aplicabilidad del Bussiness Inteligence resulta particularmente importante ya que, dentro de las entidades de gobierno autónomo no se suele priorizar proyectos de manejo y análisis de datos, priorizando en su lugar las obras e infraestructura visible, en nuestro caso vialidad, riego, etc.

En mi entorno, usualmente encontramos problemas como sistemas aislados, bases de datos heredadas y formatos incompatibles, así como la difícil interoperabilidad de sistemas y bases de datos de otras entidades y niveles de gobierno, en cuyo contexto la implementación de procesos ETL se convierten en una necesidad fundamental para consolidar una estructura de Bussiness Inteligence, permitiendo optimizar recursos de la insitución mediante la toma de decisiones mejor informadas y analizadas.

En nuestro caso como institución, podemos aplicar los procesos de BI en múltiples campos, desde el análisis de ciberseguridad como es el caso de nuestro ejercicio, hasta el análisis de procesos de contratación pública, que permitan identificar problemas comunes o cuellos de botella que retrazan la ejecución presupuestaria y a su vez permitan tomar decisiones correctivas para mejorar los procesos desde la fase de planificación y ejecución de obras.

También son aplicables estos procesos en el seguimiento de proyectos, lo que conocemos como gobierno por resultados, en el cual se realiza un análisis de la planificación inicial, plasmada en el Plan Operativo Anual con lo realmente ejecutado a nivel presupuestario y contable, situación que resulta difícil al tener sistemas que no interoperan entre sí, tanto la parte de planificación y la parte financiera que se opera dentro del sistema del Ministerio de Finanzas (eSIGEF).

Un gran desafío consiste en consolidar y transformar los datos dispersos en activos de alto valor público, dotando a la institución de la capacidad de operar de manera más inteligente, transparente y orientada al servicio del ciudadano.

In [20]:
df_csv

,incident_id,timestamp,attack_type,protocol,source_ip,destination_port,packets_sent,bytes_transferred,duration_seconds,severity,status,affected_system,analyst_assigned,false_positive
0,1,2024-01-01 00:00:00,Ransomware,HTTP,192.168.154.202,3306,25017,8071948,2,High,Blocked,Web Server,Ana Torres,False
1,2,2024-01-01 12:00:00,Brute Force,UDP,192.168.90.254,22,31295,6844556,2273,Critical,Blocked,Web Server,Luis Mora,True
2,3,2024-01-02 00:00:00,Phishing,HTTP,192.168.162.115,3306,21666,4946193,2849,Medium,Blocked,DB Server,Alejandra T.,False
3,4,2024-01-02 12:00:00,Ransomware,HTTP,192.168.105.135,21,10016,6734239,1769,Critical,Blocked,Auth Server,Luis Mora,True
4,5,2024-01-03 00:00:00,XSS,HTTPS,192.168.196.58,8080,20906,8196834,1950,High,Detected,API Gateway,Luis Mora,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,296,2024-05-27 12:00:00,XSS,UDP,192.168.243.31,80,44581,9735840,303,Medium,Mitigated,Auth Server,Luis Mora,False
296,297,2024-05-28 00:00:00,Brute Force,HTTP,192.168.40.229,3306,8602,4531913,2792,Medium,Blocked,Firewall,Ana Torres,False
297,298,2024-05-28 12:00:00,Brute Force,TCP,192.168.36.152,80,645,10168231,588,High,Blocked,Firewall,Alejandra T.,False
298,299,2024-05-29 00:00:00,XSS,UDP,192.168.223.6,80,37214,8412063,987,Medium,Detected,DB Server,Ana Torres,False


In [21]:
df_json

,vuln_id,cve_id,system,vendor,vulnerability_type,cvss_score,cvss_severity,patch_available,patch_applied,discovery_date,remediation_deadline_days,affected_hosts,exploited_in_wild
0,1,CVE-2024-29772,Apache 2.4,Apache Foundation,Denial of Service,5.2,Medium,True,True,2024-09-07,7,6,False
1,2,CVE-2024-19156,Python 3.8,PostgreSQL Global Dev Group,Buffer Overflow,3.9,Low,True,True,2024-10-19,90,4,False
2,3,CVE-2023-82963,nginx 1.18,PostgreSQL Global Dev Group,Remote Code Execution,8.9,High,True,True,2024-02-19,15,24,True
3,4,CVE-2022-18229,Python 3.8,OpenSSL Project,Information Disclosure,6.5,Medium,False,False,2024-10-15,30,20,False
4,5,CVE-2023-41994,nginx 1.18,Python Software Foundation,Information Disclosure,2.7,Low,True,False,2024-10-03,7,33,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,76,CVE-2023-19458,Apache 2.4,OpenSSL Project,Buffer Overflow,4.1,Medium,False,False,2024-03-08,15,27,False
76,77,CVE-2022-98356,Python 3.8,Apache Foundation,Privilege Escalation,3.9,Low,True,None,2024-05-12,30,48,False
77,78,CVE-2023-67592,OpenSSL 1.1,OpenSSL Project,Remote Code Execution,4.0,Medium,True,True,2024-06-03,90,17,False
78,79,CVE-2022-78984,Node.js 14,Apache Foundation,Authentication Bypass,3.9,Low,False,True,2024-01-16,15,29,False


In [22]:
df_postgres

,event_id,event_date,channel,event_type,http_method,endpoint,response_code,response_time_ms,risk_score,resolved,user_agent,country_code
0,1,2024-01-01,Internet Banking,Auth Failure,GET,/api/v1/transactions,429,1223,4,True,Mozilla/5.0,EC
1,2,2024-01-03,Cash Management,Token Expired,DELETE,/api/v1/transactions,200,344,3,False,Mozilla/5.0,CN
2,3,2024-01-05,Internet Banking,Rate Limit Exceeded,GET,/api/v2/auth,500,4944,3,True,nan,EC
3,4,2024-01-07,Cash Management,Anomalous Pattern,DELETE,/api/v1/balance,429,2459,2,True,nan,EC
4,5,2024-01-09,Internet Banking,Token Expired,DELETE,/api/v1/transactions,400,3321,2,False,Mozilla/5.0,EC
...,...,...,...,...,...,...,...,...,...,...,...,...
145,146,2024-10-17,Cash Management,Anomalous Pattern,DELETE,/api/v1/transactions,400,4919,4,False,PostmanRuntime/7.x,CN
146,147,2024-10-19,ATM Network,Injection Attempt,DELETE,/api/v1/balance,403,1704,3,True,Mozilla/5.0,CO
147,148,2024-10-21,POS Terminal,Auth Failure,DELETE,/api/v1/transfer,401,3962,5,False,Python-requests/2.x,EC
148,149,2024-10-23,Mobile Banking,Unauthorized Endpoint,GET,/api/v1/transfer,400,1965,1,True,PostmanRuntime/7.x,CO
